# 02 Data Cleaning and Integration

End-to-end view of the cleaning pipeline: dirty raw data → standardized processed tables.

> Run the pipeline first: `python run_pipeline.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

raw = ROOT / 'data' / 'raw' / 'sample'
interim = ROOT / 'data' / 'interim'

dirty = pd.read_csv(raw / 'dirty_reactors_raw.csv')
cleaned = pd.read_csv(processed / 'reactors_master.csv')
print(f"Raw rows: {len(dirty)}  →  Cleaned rows: {len(cleaned)}")
print(f"Raw columns: {len(dirty.columns)}  →  Cleaned columns: {len(cleaned.columns)}")

## What was dirty in the raw data

In [ ]:
print("=== Sample of raw data (first 5 rows, key columns) ===")
dirty[['reactor_name','status','net_capacity_mwe','reactor_type']].head(8)

In [ ]:
# Specific dirty issues baked in by sample_data.py
print("Status values in raw data (inconsistent casing/spacing):")
print(dirty['status'].value_counts().to_string())
print()
print("net_capacity_mwe dtype:", dirty['net_capacity_mwe'].dtype)
print("net_capacity_mwe sample values:", dirty['net_capacity_mwe'].dropna().unique()[:5])
print()
print("Duplicate rows:", dirty.duplicated().sum())
print("Null net_capacity:", dirty['net_capacity_mwe'].isna().sum())

## What the cleaning pipeline fixed

In [ ]:
print("=== Status normalization ===")
print(cleaned['status_group'].value_counts().to_string())
print()
print("=== Types added by cleaning ===")
new_cols = [c for c in cleaned.columns if c not in dirty.columns]
print(new_cols)
print()
print("=== Duplicate reactor_ids after cleaning ===", cleaned.duplicated('reactor_id').sum())
print("=== Nulls in capacity_mwe ===", cleaned['capacity_mwe'].isna().sum())

## Schema comparison: raw vs processed

In [ ]:
schema = pd.DataFrame({
    'raw': [dirty[c].dtype if c in dirty.columns else '—' for c in cleaned.columns],
    'processed': [cleaned[c].dtype for c in cleaned.columns]
}, index=cleaned.columns)
schema[schema['raw'] != '—']

## Feature engineering layers

In [ ]:
print("Columns added by reactor_features.py (flag columns):")
flag_cols = [c for c in cleaned.columns if c.endswith('_flag')]
print(flag_cols)

print()
print("Columns added by pipeline_features.py (scoring):")
pipeline = pd.read_csv(processed / 'reactor_pipeline.csv')
score_cols = ['project_maturity_score','delay_risk_score','realization_probability','expected_operation_year']
print(pipeline[score_cols].describe().round(2))

## Data lineage summary

In [ ]:
print("""
Raw (dirty_reactors_raw.csv)
    ↓ clean_reactor_data.py
Interim (reactors_cleaned.csv)
    ↓ add_reactor_feature_flags()
    ↓ build_technology_taxonomy()
    ↓ build_reactor_pipeline()
    ↓ build_country_profiles()
    ↓ build_capacity_scenarios()
    ↓ integrate_all()
Processed (reactors_master.csv, reactor_pipeline.csv, country_nuclear_profile.csv, ...)
    ↓ ML models, NLP, SQL
Outputs (predictions/, metrics/, reports/)
""")

## Source confidence scores

In [ ]:
conf = cleaned.groupby('source_name')['source_confidence'].agg(['mean','count']).round(2)
conf.columns = ['avg_confidence','records']
print(conf)

fig, ax = plt.subplots(figsize=(9,4))
sns.barplot(data=conf.reset_index(), x='source_name', y='avg_confidence', ax=ax, palette='Blues_d')
ax.set_ylim(0,1); ax.set_title('Source Confidence by Source Name')
ax.set_xlabel(''); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()